# Session 4 · First Model, Full Workflow

**Machine Learning Foundations · Sanketana School of Code**

Today you put it all together. Across three sessions you learned the moves; now you run the whole thing yourself, start to finish — and we finally give the workflow its name:

```
data  →  model  →  evaluation  →  insight
```

By the end of this notebook you will have **trained a model end-to-end** and be able to:

- carry a clean dataset through all four steps yourself
- explain why a clean dataset skips the cleaning and encoding of Sessions 2–3
- read the train-vs-test scores as evidence of learning, not memorizing
- produce one **insight** — change a habit, watch the prediction move, and say what it means

**How this notebook works:** this is *your* run. ✏️ cells are yours, and today there are more of them. Working code is provided so nothing breaks, but try to type the key lines yourself.

## Warm-up · Last session's homework

Your coach will walk through Session 3's laptop homework with you (about 10 minutes). Check: did your `fit` / `predict` / `score` trio run, and did you read **both** scores?

Today you'll use that exact trio — but you drive, and we add a fourth step the model alone can't give you: **insight**.

## The workflow, named

Keep this on screen. Every part of today's notebook is one of these four words:

| Step | What it means | Today |
|---|---|---|
| **data** | get it, look at it, make it model-ready | load `student_habits.csv`, split into `X` and `y` |
| **model** | pick a model, let it learn the pattern | `train_test_split`, then `fit` |
| **evaluation** | how well did it do, *honestly*? | `score` on held-out data |
| **insight** | what does this tell a human? | nudge a habit, read the change |

Today's dataset is the **anchor** dataset — 420 students, their habits, and their test scores. You'll meet this same cohort again in Modules 2, 3, and 5, each time through a different lens.

## Step 1 · DATA

Load the data and look. Notice what's *not* here: no missing values, no text columns. This dataset came clean — so the cleaning and encoding work from Sessions 2–3 simply isn't needed today. That's the point: the spotlight is on the workflow.

In [ ]:
import pandas as pd

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")

print("shape:", students.shape)
print("missing values:", students.isna().sum().sum())
students.head()

In [ ]:
# Look at the numbers. All clean, all numeric (except the ID).
students.describe().round(1)

Now the one bit of prep that's always needed: split the table into **`X`** (features the model may look at) and **`y`** (the label we want to predict).

We predict `test_score`. We drop two columns from `X`:
- `student_id` — an ID identifies a student but says nothing about their score (the `title` trap from Session 1).
- `passed` — it's derived straight from `test_score`, so leaving it in would let the model "cheat" by peeking at a near-copy of the answer.

In [ ]:
# ✏️ TODO: define y (the label) and X (the features). (Working version provided.)
y = students["test_score"]
X = students.drop(columns=["student_id", "test_score", "passed"])

print("y (label):", y.name)
print("X (features):", list(X.columns))
print("X shape:", X.shape)

## Step 2 · MODEL

Split off a hidden test set first (the memorizing check), then `fit` the model on the training data. Same black-box `LinearRegression` as last session — we run the move and read the result; *how* it works is Module 2.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# ✏️ TODO: hide 20% as a test set, then fit on the training set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("training rows:", X_train.shape[0], " | hidden test rows:", X_test.shape[0])

model = LinearRegression()
model.fit(X_train, y_train)
print("model trained.")

## Step 3 · EVALUATION

How well did it do — *honestly*? Score on the training data (already seen) **and** on the hidden test data (never seen). For a regressor, `.score()` is R²: higher is better, 1.0 is perfect. (We read it; we unpack it in Session 8.)

In [ ]:
# ✏️ TODO: score on training data and on the hidden test data.
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("score on training data (already seen):", round(train_score, 3))
print("score on test data (never seen):      ", round(test_score, 3))

### ✏️ Read the two scores

1. Are the two scores **close**? (They might be in either order by a hair — what matters is the size of the gap.)
2. The Session 1 lookup table scored a perfect 100% on data it had memorized and was useless on anything new. In one sentence: how do two *close* scores here show the opposite — that this model **learned** rather than memorized?

*Your answers:*

1.
2.

## Step 4 · INSIGHT

A score is not the point — *understanding* is. This is the step the model can't do for you.

The trick: predict a scenario, then **change one thing**. We'll take one student, predict their score, then ask "what if they studied 5 more hours a week?" and predict again.

In [ ]:
# Take one real student as our starting scenario.
scenario = X_test.iloc[[0]].copy()
print("scenario student's habits:")
print(scenario.to_string(index=False))

base_prediction = model.predict(scenario)[0]
print(f"\npredicted test score: {base_prediction:.1f}")

In [ ]:
# What if this student studied 5 more hours per week? Change ONE feature, predict again.
more_study = scenario.copy()
more_study["study_hours_per_week"] = more_study["study_hours_per_week"] + 5
study_prediction = model.predict(more_study)[0]

# And separately: what if they cut screen time by 2 hours a day?
less_screen = scenario.copy()
less_screen["screen_time_hours_per_day"] = less_screen["screen_time_hours_per_day"] - 2
screen_prediction = model.predict(less_screen)[0]

print(f"base prediction:          {base_prediction:5.1f}")
print(f"+5 study hours/week:      {study_prediction:5.1f}   (change: {study_prediction - base_prediction:+.1f})")
print(f"-2 screen hours/day:      {screen_prediction:5.1f}   (change: {screen_prediction - base_prediction:+.1f})")

### ✏️ Write the insight

Look at the two changes above.

1. Which habit change moved the predicted score **more**? Write one plain-English sentence a parent could understand — starting with *"In this data, ..."*
2. Why must we say *"in this data"* and not *"studying causes higher scores"*? (Think about what the model actually found, and who the 420 students were.)

*Your insight:*

1.
2.

### ✏️ Stretch — your own "what if"

Invent your own scenario change on `scenario` — for example, drop `attendance_pct` to 60, or raise `practice_sessions_per_week`. Predict, compare to the base, and interpret it in one sentence.

In [ ]:
# ✏️ TODO: your own "what if". Change one feature, predict, compare to base_prediction.
my_scenario = scenario.copy()
my_scenario["attendance_pct"] = 60          # <- change this line to try your own
my_prediction = model.predict(my_scenario)[0]

print(f"base:        {base_prediction:5.1f}")
print(f"my scenario: {my_prediction:5.1f}   (change: {my_prediction - base_prediction:+.1f})")

## What we learned

✏️ Three quick reflections — one line each:

1. The four steps of the workflow, from memory:
2. The one habit that moved the predicted score most, in this data:
3. One reason the model could be wrong about a real student:

---

**You trained a model today — end to end.** You walked the whole path: **data → model → evaluation → insight.** That sentence is the spine of everything that follows. Every model in this course, however fancy, is run along these same four steps.

**Next: Module 2.** We've been treating the model as a sealed box. Now we open it: how does linear regression actually draw its line through the data, and which habits matter most? The black box becomes glass.

**Homework:** `homework.ipynb` — the full workflow, on your own, on the *messy* housing data from Sessions 2–3. It states its success criterion at the top. Revise with `explainer.md`.